In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')

In [ ]:
%pip install "transformers>=4.51.0,<5.0.0" sentence-transformers --upgrade --quiet

In [ ]:
pip show sentence-transformers transformers | grep -E "Name|Version"

In [ ]:
import torch, transformers, sentence_transformers
print(torch.__version__)
print(transformers.__version__)      # doit être 4.51.x ou 4.52.x
print(sentence_transformers.__version__)

# Imports

In [ ]:
import os, re, sys
sys.path.insert(0, "..")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import glob
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login

from candidates.bm25_retriever import BM25Retriever
from candidates.dense_retriever import DenseRetriever
from utils.prompt_builder import build_prompt_zero_shot, build_prompt_mcq
from utils.response_parser import parse_response
from utils.metrics import is_correct, recall_at_k, compute_metrics, print_metrics


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
print(HF_TOKEN is not None)

In [ ]:
login(token=HF_TOKEN)

In [ ]:
MODE = "mcq_bm25"
MODEL_NAME = "Qwen/Qwen3-4B"
K = 100
MAX_EXAMPLES = None      
BATCH_SIZE = 64

TITLES_PATH = "../data/wikidata_qid_titles.csv"
INDEX_PATH = "../candidates/index/bm25.pkl"

DENSE_INDEX_DIR = "../candidates/index/"  
DENSE_MODEL     = "Qwen/Qwen3-Embedding-4B"

BENCHMARKS = "../data/benchmarks"
RESULTS_DIR = "../evaluation/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ELGOLD Dataset

Files `.txt` with annotations `{{mention|TYPE|gold_title}}`.
We extract each mention + context (150 chars from each side).

In [ ]:
ANNOT_RE = re.compile(r"\{\{([^|{}]+)\|([^|{}]+)\|([^{}]+)\}\}")

def parse_elgold_file(path, context_window=150):
    raw = Path(path).read_text(encoding="utf-8")
    clean = ANNOT_RE.sub(lambda m: m.group(1).strip(), raw)
    examples, cursor, last_end = [], 0, 0
    for match in ANNOT_RE.finditer(raw):
        mention = match.group(1).strip()
        ent_type = match.group(2).strip()
        gold = match.group(3).strip()
        cursor += len(raw[last_end:match.start()])
        end = cursor + len(mention)
        snippet = clean[max(0, cursor-context_window):min(len(clean), end+context_window)]
        examples.append({"file": Path(path).name, "mention": mention,
                          "entity_type": ent_type, "gold": gold, "context": snippet})
        cursor, last_end = end, match.end()
    return examples

rows = []
for f in glob.glob(f"{BENCHMARKS}/*.txt"):
    rows.extend(parse_elgold_file(f))

df = pd.DataFrame(rows).drop_duplicates()
if MAX_EXAMPLES:
    df = df.head(MAX_EXAMPLES)

print(f"Exemples : {len(df)}")
print(f"Fichiers : {df['file'].nunique()}")
print(f"Types : {df['entity_type'].value_counts().to_dict()}")
df.head(3)

# Candidate generation

## Sparse retrieval (BM25)

### BM25 index

The index is built the first time (only once) from `wikidata_qid_titles.csv` and saved as `.pkl`.  
The following times, we load it from the `.pkl`  

The index contains Wikipedia titles (candidate entities).  

In [ ]:
if Path(INDEX_PATH).exists():
    print("Index found → loading...")
    retriever = BM25Retriever.load(INDEX_PATH)
else:
    print("Index not found → construction...")
    titles_df = pd.read_csv(TITLES_PATH)
    print(f"Columns : {titles_df.columns.tolist()}")

    titles = titles_df["title"].dropna().tolist()
    print(f"{len(titles):,} titles loaded")
    retriever = BM25Retriever.from_entity_list(titles)
    retriever.save(INDEX_PATH)

print(f"Index ready : {len(retriever.entity_titles):,} entities")

BM25 verification (small test)

In [ ]:
for _, row in df.head(5).iterrows():
    candidates = retriever.retrieve(row["mention"], k=K)
    gold_found = "✓" if recall_at_k(row["gold"], candidates) else "✗"
    print(f"{gold_found} mention='{row['mention']}'  gold='{row['gold']}'")
    print(f"  → {candidates}\n")

### Recall@K on benchmark

Recall@K = pourcentage d'exemples où le gold est dans les K candidats BM25.  

L'accuracy ne peut pas le dépasser.

In [ ]:
# Generate candidates for all the benchmark
df["candidates"] = df["mention"].apply(lambda m: retriever.retrieve(m, k=K))
df["gold_in_candidates"] = df.apply(
    lambda row: recall_at_k(row["gold"], row["candidates"]), axis=1
)

recall = df["gold_in_candidates"].mean()
print(f"Recall@{K} (BM25) : {recall:.2%}")

missed = df[~df["gold_in_candidates"]]
print(f"\nGold absent des candidats : {len(missed)}/{len(df)} exemples")
print("Exemples ratés par BM25 :")
missed[["mention", "gold", "candidates"]].head(5)

## Dense retrieval

In [ ]:
dense_index_exists = (
    os.path.exists(f"{DENSE_INDEX_DIR}/dense.faiss") and
    os.path.exists(f"{DENSE_INDEX_DIR}/dense_meta.pkl")
)
 
if dense_index_exists:
    print("Dense index found → loading ...")
    dense_retriever = DenseRetriever.load(DENSE_INDEX_DIR)
else:
    print("Dense index not found → building ...")

    # liste déjà chargée dans retriever.entity_titles (BM25)
    titles = retriever.entity_titles  
    dense_retriever = DenseRetriever(model_name=DENSE_MODEL)
    dense_retriever.build_index(titles, batch_size=64)
    dense_retriever.save(DENSE_INDEX_DIR)
 
print(f"Dense index ready : {dense_retriever.index.ntotal:,} vectors")

Dense retrieval check

In [ ]:
for _, row in df.head(5).iterrows():
    candidates_d = dense_retriever.retrieve(row["mention"], k=K)
    found = "✓" if recall_at_k(row["gold"], candidates_d) else "✗"
    print(f"{found}  '{row['mention']}'  →  gold='{row['gold']}'")
    print(f"     {candidates_d}")
    print()

## Recall@K

In [ ]:
def fuse_candidates(bm25_cands: list[str], dense_cands: list[str], k: int = 10) -> list[str]:
    seen, merged = set(), []
    for b, d in zip(bm25_cands, dense_cands):
        for c in (b, d):
            if c.lower() not in seen:
                seen.add(c.lower())
                merged.append(c)
            if len(merged) == k:
                return merged
    # If one list is shorter, drain the other
    for c in (bm25_cands + dense_cands):
        if c.lower() not in seen:
            seen.add(c.lower())
            merged.append(c)
        if len(merged) == k:
            return merged
    return merged[:k]

In [ ]:
# Generate candidates for all 3 modes
df["candidates_bm25"] = df["mention"].apply(lambda m: retriever.retrieve(m, k=K))
df["candidates_dense"] = df["mention"].apply(lambda m: dense_retriever.retrieve(m, k=K))
df["candidates_fusion"] = df.apply(
    lambda row: fuse_candidates(row["candidates_bm25"], row["candidates_dense"], k=K),
    axis=1
)

In [ ]:
# Recall@K for each
recall_bm25 = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_bm25"]), axis=1).mean()
recall_dense = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_dense"]), axis=1).mean()
recall_fusion = df.apply(lambda r: recall_at_k(r["gold"], r["candidates_fusion"]), axis=1).mean()
 
print(f"Recall@{K}  BM25    : {recall_bm25:.2%}")
print(f"Recall@{K}  Dense   : {recall_dense:.2%}")
print(f"Recall@{K}  Fusion  : {recall_fusion:.2%}  ← expected best")
print()
print("Plafonds MCQ (le LLM ne peut pas dépasser ces chiffres) :")
print(f"  BM25   → max accuracy = {recall_bm25:.2%}")
print(f"  Dense  → max accuracy = {recall_dense:.2%}")
print(f"  Fusion → max accuracy = {recall_fusion:.2%}")

# Evaluation

## Loading model

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
).eval()
print(f"Loaded on : {next(model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████████████| 3/3 [00:45<00:00, 15.14s/it]


Loaded on : cuda:0


In [14]:
def generate_batch(prompts, max_new_tokens=20):
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    return [
        tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        for out in outputs
    ]

In [15]:
def run_eval_mode(df_in, candidates_col, mode_label):
    results = []
    records = df_in.copy()
    records["candidates"] = records[candidates_col]
    records["gold_in_candidates"] = records.apply(
        lambda r: recall_at_k(r["gold"], r["candidates"]), axis=1
    )
 
    for i in tqdm(range(0, len(records), BATCH_SIZE), desc=mode_label):
        batch = records.iloc[i : i + BATCH_SIZE].to_dict("records")
        prompts = [
            build_prompt_mcq(r["context"], r["mention"], r["entity_type"], r["candidates"])
            for r in batch
        ]
        try:
            raws = generate_batch(prompts)
        except Exception as e:
            for r in batch:
                results.append({**r, "predicted": "", "correct": False,
                                 "raw_output": "", "error": str(e)})
            continue
 
        for rec, raw in zip(batch, raws):
            try:
                parsed = parse_response(raw, mode="mcq", candidates=rec["candidates"])
                pred, error = parsed["predicted"], ""
            except Exception as e:
                pred, error = "", str(e)
 
            results.append({
                "mention": rec["mention"],
                "entity_type": rec["entity_type"],
                "gold": rec["gold"],
                "candidates": rec["candidates"],
                "gold_in_candidates":rec["gold_in_candidates"],
                "predicted": pred,
                "correct": is_correct(pred, rec["gold"]) if not error else False,
                "raw_output": raw,
                "error": error,
                "retrieval_mode": mode_label,
            })
 
    return pd.DataFrame(results)
 

In [16]:
slug = MODEL_NAME.replace("/", "_")
 
df_bm25   = run_eval_mode(df, "candidates_bm25", "mcq_bm25")
df_dense  = run_eval_mode(df, "candidates_dense", "mcq_dense")
df_fusion = run_eval_mode(df, "candidates_fusion", "mcq_fusion")
 
# Save
for df_r, label in [(df_bm25, "mcq_bm25"), (df_dense, "mcq_dense"), (df_fusion, "mcq_fusion")]:
    df_r.to_csv(f"{RESULTS_DIR}/{label}_{slug}.csv", index=False)

KeyError: 'candidates_bm25'

In [ ]:
import pandas as pd
 
rows = []
for df_r, label, recall in [
    (df_bm25, "BM25", recall_bm25),
    (df_dense, "Dense", recall_dense),
    (df_fusion, "Fusion", recall_fusion),
]:
    m = compute_metrics(df_r)
    rows.append({
        "Retriever": label,
        "Recall@K": f"{recall:.2%}",
        "Accuracy": f"{m['accuracy']:.2%}",
        "LLM loss": f"-{(recall - m['accuracy']):.2%}",
        "n_correct": m["n_correct"],
        "n_total": m["n_total"],
    })
 
df_comparison = pd.DataFrame(rows)
print(f"\nModel : {MODEL_NAME}")
print(df_comparison.to_string(index=False))

## Test on examples

In [17]:
for rec in df.head(3).to_dict("records"):
    prompt = (
        build_prompt_zero_shot(rec["context"], rec["mention"], rec["entity_type"])
        if MODE == "zero_shot"
        else build_prompt_mcq(rec["context"], rec["mention"], rec["entity_type"], rec["candidates"])
    )
    raw = generate_batch([prompt])[0]
    parsed = parse_response(raw, mode="zero_shot" if MODE == "zero_shot" else "mcq",
                            candidates=rec["candidates"])
    ok = is_correct(parsed["predicted"], rec["gold"])
    print(f"{'✓' if ok else '✗'}  mention='{rec['mention']}'")
    print(f"   gold : {rec['gold']}")
    print(f"   raw output: {repr(raw)}")
    print(f"   predicted : {parsed['predicted']}")
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✗  mention='Hollywood'
   gold : Cinema of the United States
   raw output: 'Answer: A\n\nThe mention "Hollywood" in the text refers to the real-world entity "'
   predicted : Hollywood

✗  mention='Hellbound'
   gold : The Hellbound Heart
   raw output: 'Answer: C\nThe mention "Hellbound" in the text refers to "Hellbound: Hell'
   predicted : Hellbound: Hellraiser II

✓  mention='Constantine'
   gold : Constantine (film)
   raw output: 'Answer: N\n\nThe mention "Constantine" in the text refers to the film "Constantine'
   predicted : Constantine (film)



## Zero-shot Evaluation

In [18]:
results  = []
records  = df.to_dict("records")
llm_mode = "zero_shot" if MODE == "zero_shot" else "mcq"

for i in tqdm(range(0, len(records), BATCH_SIZE)):
    batch = records[i : i + BATCH_SIZE]
    prompts = [
        build_prompt_zero_shot(r["context"], r["mention"], r["entity_type"])
        if MODE == "zero_shot"
        else build_prompt_mcq(r["context"], r["mention"], r["entity_type"], r["candidates"])
        for r in batch
    ]
    try:
        raws = generate_batch(prompts)
    except Exception as e:
        for r in batch:
            results.append({**r, "predicted": "", "correct": False, "raw_output": "", "error": str(e)})
        continue

    for rec, raw in zip(batch, raws):
        try:
            parsed = parse_response(raw, mode=llm_mode, candidates=rec["candidates"])
            pred, error = parsed["predicted"], ""
        except Exception as e:
            pred, error = "", str(e)

        results.append({
            "file": rec["file"],
            "mention": rec["mention"],
            "entity_type": rec["entity_type"],
            "gold": rec["gold"],
            "candidates": rec["candidates"],
            "gold_in_candidates":rec["gold_in_candidates"],
            "predicted": pred,
            "correct": is_correct(pred, rec["gold"]) if not error else False,
            "raw_output": raw,
            "error": error,
        })

df_results = pd.DataFrame(results)
slug = MODEL_NAME.replace("/", "_")
out_path = f"{RESULTS_DIR}/{MODE}_{slug}.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved : {out_path}")

100%|████████████████████████████████████████████████████████████████| 49/49 [32:42<00:00, 40.04s/it]


Saved : ../evaluation/results/mcq_bm25_Qwen_Qwen3-4B.csv


## Results

In [19]:
metrics = compute_metrics(df_results)
print_metrics(metrics, model_name=f"{MODE} / {MODEL_NAME}")

if MODE == "mcq_bm25":
    print(f"\nRecall@{K} BM25 (plafond) : {recall:.2%}")
    print(f"Gap plafond → accuracy : -{(recall - metrics['accuracy']):.2%}  (LLM errors)")

# Errors: when gold was among candidates but the LLM did not select it
ok_retrieval = df_results[df_results["gold_in_candidates"]] if "gold_in_candidates" in df_results.columns else df_results
llm_errors = ok_retrieval[~ok_retrieval["correct"] & (ok_retrieval["error"] == "")]
print(f"\nErreurs LLM (gold était dans les candidats) : {len(llm_errors)}")
for _, r in llm_errors.head(5).iterrows():
    print(f"  '{r['mention']}' | gold='{r['gold']}' | pred='{r['predicted']}' | raw={repr(r['raw_output'])}")


================================================== [mcq_bm25 / Qwen/Qwen3-4B]
  Accuracy   : 19.15%  (435/2271)
  Recall@K   : 56.14%  (gold in candidates)
  Errors     : 832 skipped

Recall@15 BM25 (plafond) : 55.75%
Gap plafond → accuracy : -36.60%  (LLM errors)

Erreurs LLM (gold était dans les candidats) : 840
  'Global Ratings' | gold='S&P Global Ratings' | pred='' | raw='The answer should be in the format specified, with the correct letter corresponding to the correct entity.\nAnswer'
  'Hong Kong' | gold='Hong Kong' | pred='Apm (Hong Kong)' | raw='To solve this quote, I need to determine which entity the term [Hong Kong] refers to in'
  'China' | gold='China' | pred='Mainland China' | raw='... (the rest of the text is irrelevant)\nAnswer: B\nThe mention "China" in'
  'China' | gold='China' | pred='' | raw='1. The mention "China" in the text refers to the country, as it is discussing the'
  'Japan' | gold='Japan' | pred='' | raw='the text mentions "Japan" in the context of a cou